# Signal health notebook - 9 key analyses

This notebook adds nine high-value analyses for the `ngrams` stream:
1) Dataset coverage snapshot
2) Ingestion velocity over time
3) Source contribution trends
4) Language mix and drift
5) n-gram mix by source
6) Phrase concentration (repeat pressure)
7) Novelty rate (new phrases)
8) Cross-source phrase overlap
9) Recency and staleness by source

In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import psycopg
from dotenv import load_dotenv

load_dotenv(Path('..') / '.env')

conninfo = psycopg.conninfo.make_conninfo(
    host=os.environ['POSTGRES_HOST'],
    port=int(os.getenv('POSTGRES_PORT', '5432')),
    user=os.environ['POSTGRES_USER'],
    password=os.environ['POSTGRES_PASSWORD'],
    dbname=os.environ['POSTGRES_DATABASE'],
    sslmode=os.getenv('PGSSLMODE', 'require'),
)

def q(sql: str, params: dict | None = None) -> pd.DataFrame:
    with psycopg.connect(conninfo) as conn, conn.cursor() as cur:
        cur.execute(sql, params or None)
        cols = [d.name for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)

print('Ready - using DB host:', os.environ['POSTGRES_HOST'])

## 1) Dataset coverage snapshot
Quick sanity check for scale, time span, and dimensionality.

In [ ]:
coverage = q('''
SELECT
    COUNT(*)                  AS rows_total,
    COUNT(DISTINCT source)    AS sources,
    COUNT(DISTINCT language)  AS languages,
    COUNT(DISTINCT words)     AS distinct_phrases,
    MIN("timestamp")         AS first_seen,
    MAX("timestamp")         AS last_seen
FROM public.ngrams
''')
coverage

## 2) Ingestion velocity over time
Daily event volume to detect spikes, dropouts, and trend shifts.

In [ ]:
daily = q('''
SELECT
    date_trunc('day', "timestamp") AS day,
    COUNT(*) AS events
FROM public.ngrams
GROUP BY 1
ORDER BY 1
''')

fig = px.line(daily, x='day', y='events', title='Daily ingestion velocity')
fig.update_layout(height=420, xaxis_title='', yaxis_title='events')
fig.show()

## 3) Source contribution trends
How each source contributes to total volume over time.

In [ ]:
src_daily = q('''
WITH top_sources AS (
    SELECT source
    FROM public.ngrams
    GROUP BY source
    ORDER BY COUNT(*) DESC
    LIMIT 8
)
SELECT
    date_trunc('day', n."timestamp") AS day,
    n.source,
    COUNT(*) AS events
FROM public.ngrams n
JOIN top_sources t ON t.source = n.source
GROUP BY 1, 2
ORDER BY 1, 3 DESC
''')

fig = px.area(src_daily, x='day', y='events', color='source',
              title='Top source contribution by day')
fig.update_layout(height=460, xaxis_title='', yaxis_title='events')
fig.show()

## 4) Language mix and drift
Language share by day to identify market or source-mix shifts.

In [ ]:
lang_daily = q('''
SELECT
    date_trunc('day', "timestamp") AS day,
    language,
    COUNT(*) AS events
FROM public.ngrams
GROUP BY 1, 2
ORDER BY 1
''')

totals = lang_daily.groupby('day', as_index=False)['events'].sum().rename(columns={'events': 'total'})
lang_share = lang_daily.merge(totals, on='day', how='left')
lang_share['share'] = lang_share['events'] / lang_share['total']

fig = px.area(lang_share, x='day', y='share', color='language', groupnorm='fraction',
              title='Language share over time')
fig.update_layout(height=460, xaxis_title='', yaxis_title='share')
fig.show()

## 5) n-gram mix by source
Structural content fingerprint: which sources emit short vs. longer phrases.

In [ ]:
ng_by_source = q('''
WITH top_sources AS (
    SELECT source
    FROM public.ngrams
    GROUP BY source
    ORDER BY COUNT(*) DESC
    LIMIT 10
)
SELECT n.source, n.n_gram, COUNT(*) AS events
FROM public.ngrams n
JOIN top_sources t ON t.source = n.source
GROUP BY 1, 2
ORDER BY 1, 2
''')

pivot = ng_by_source.pivot(index='source', columns='n_gram', values='events').fillna(0)
row_totals = pivot.sum(axis=1)
norm = pivot.div(row_totals, axis=0)

fig = px.imshow(norm, aspect='auto',
                title='Relative n-gram profile by source',
                labels={'x': 'n_gram', 'y': 'source', 'color': 'share'})
fig.update_layout(height=500)
fig.show()

## 6) Phrase concentration (repeat pressure)
What fraction of volume is carried by the top phrases. High concentration can indicate redundancy.

In [ ]:
top_words = q('''
SELECT words, COUNT(*) AS freq
FROM public.ngrams
GROUP BY words
ORDER BY freq DESC
LIMIT 50
''')

total_events = int(coverage.loc[0, 'rows_total'])
top_words['share'] = top_words['freq'] / total_events
top_words['cum_share'] = top_words['share'].cumsum()

fig = go.Figure()
fig.add_trace(go.Bar(x=top_words.index + 1, y=top_words['share'], name='share per rank'))
fig.add_trace(go.Scatter(x=top_words.index + 1, y=top_words['cum_share'],
                         mode='lines+markers', name='cumulative share', yaxis='y2'))
fig.update_layout(
    title='Phrase concentration curve (top 50 phrases)',
    xaxis_title='phrase rank',
    yaxis=dict(title='share'),
    yaxis2=dict(title='cumulative share', overlaying='y', side='right'),
    height=430,
)
fig.show()

top_words.head(15)

## 7) Novelty rate (new phrases)
Daily fraction of phrases that are first-ever sightings in the stream.

In [ ]:
novelty = q('''
WITH first_seen AS (
    SELECT words, MIN("timestamp") AS first_ts
    FROM public.ngrams
    GROUP BY words
),
daily_total AS (
    SELECT date_trunc('day', "timestamp") AS day, COUNT(*) AS total_events
    FROM public.ngrams
    GROUP BY 1
),
daily_new AS (
    SELECT date_trunc('day', first_ts) AS day, COUNT(*) AS new_phrases
    FROM first_seen
    GROUP BY 1
)
SELECT
    t.day,
    t.total_events,
    COALESCE(n.new_phrases, 0) AS new_phrases,
    COALESCE(n.new_phrases, 0)::float / NULLIF(t.total_events, 0) AS novelty_rate
FROM daily_total t
LEFT JOIN daily_new n USING(day)
ORDER BY t.day
''')

novelty['novelty_rate_7d'] = novelty['novelty_rate'].rolling(7, min_periods=1).mean()
fig = px.line(novelty, x='day', y=['novelty_rate', 'novelty_rate_7d'],
              title='Daily novelty rate (with 7-day smoothing)')
fig.update_layout(height=430, xaxis_title='', yaxis_title='rate')
fig.show()

## 8) Cross-source phrase overlap
Jaccard similarity of vocabularies between top sources (last 30 days).

In [ ]:
overlap = q('''
WITH recent AS (
    SELECT *
    FROM public.ngrams
    WHERE "timestamp" >= NOW() - INTERVAL '30 day'
),
top_sources AS (
    SELECT source
    FROM recent
    GROUP BY source
    ORDER BY COUNT(*) DESC
    LIMIT 8
),
vocab AS (
    SELECT DISTINCT source, words
    FROM recent
    WHERE source IN (SELECT source FROM top_sources)
),
pairs AS (
    SELECT a.source AS source_a, b.source AS source_b
    FROM top_sources a
    CROSS JOIN top_sources b
    WHERE a.source <= b.source
),
sizes AS (
    SELECT source, COUNT(*) AS vocab_size
    FROM vocab
    GROUP BY source
),
intersections AS (
    SELECT v1.source AS source_a, v2.source AS source_b, COUNT(*) AS inter_size
    FROM vocab v1
    JOIN vocab v2 ON v1.words = v2.words
    WHERE v1.source <= v2.source
    GROUP BY 1, 2
)
SELECT
    p.source_a,
    p.source_b,
    COALESCE(i.inter_size, 0) AS inter_size,
    sa.vocab_size AS size_a,
    sb.vocab_size AS size_b,
    COALESCE(i.inter_size, 0)::float
        / NULLIF((sa.vocab_size + sb.vocab_size - COALESCE(i.inter_size, 0)), 0) AS jaccard
FROM pairs p
JOIN sizes sa ON sa.source = p.source_a
JOIN sizes sb ON sb.source = p.source_b
LEFT JOIN intersections i ON i.source_a = p.source_a AND i.source_b = p.source_b
ORDER BY 1, 2
''')

heat = overlap.pivot(index='source_a', columns='source_b', values='jaccard').fillna(0)
fig = px.imshow(heat, aspect='auto',
                title='Source vocabulary overlap (Jaccard, last 30 days)',
                labels={'color': 'jaccard'})
fig.update_layout(height=520)
fig.show()

## 9) Recency and staleness by source
How fresh is each source right now?

In [ ]:
staleness = q('''
SELECT
    source,
    COUNT(*) AS events,
    MAX("timestamp") AS latest_ts,
    EXTRACT(EPOCH FROM (NOW() - MAX("timestamp"))) / 3600.0 AS lag_hours
FROM public.ngrams
GROUP BY source
ORDER BY lag_hours DESC
''')

fig = px.bar(staleness.head(20), x='source', y='lag_hours', color='events',
             title='Source staleness (hours since last event)',
             labels={'lag_hours': 'hours since last event'})
fig.update_layout(height=460, xaxis_title='', yaxis_title='hours')
fig.show()

staleness.head(20)